# 🩺 TriageCore v2: Google Colab GPU Eğitimi, Google Drive `AI` Yedekleme & Ngrok Uzak Sunucu

Bu notebook, **TriageCore** projesini yerel bilgisayarınızı hiç yormadan tamamen **Google Colab GPU (A100 / L4 / T4)** üzerinde eğitmek, eğitilen tüm ağırlıkları ve sohbet kayıtlarını **Google Drive `/MyDrive/AI/TriageCore`** klasörüne kalıcı olarak kaydetmek ve son hücrede **Ngrok / Cloudflared HTTPS Tüneli** açarak kendi Streamlit arayüzünüzden canlı kullanmak için tasarlanmıştır.

---

### 1. Adım (İlk Hücre): Google Drive Bağlantısı ve `/MyDrive/AI/TriageCore` Klasör Kurulumu
En başta Google Drive'ı bağlıyoruz. Böylece eğitim sırasındaki her checkpoint, final LoRA ağırlıkları ve sohbet logları doğrudan Drive'daki **`AI/TriageCore`** klasörüne yazılır. Eğer daha önce eğitim yaptıysanız modeli tekrar eğitmenize gerek kalmaz!

In [2]:
# Isletim sistemi ve dosya yolu islemleri icin os ve pathlib modulleri yuklenir
import os
from pathlib import Path

# Google Colab ortaminda olup olmadigimizi kontrol ederek Drive baglantisini baslatir
try:
    from google.colab import drive

    # Eger Drive zaten bagliysa (ornegin sol paneldeki klasor ikonundan baglandiysa) tekrar sorma
    if not Path("/content/drive/MyDrive").exists():
        try:
            drive.mount("/content/drive", force_remount=False)
        except Exception as e:
            print(f"[!] Standart drive.mount uyarisi: {e}")
            print("[*] 'auth.authenticate_user()' ile alternatif Google kimlik dogrulamasi deneniyor...")
            try:
                from google.colab import auth
                auth.authenticate_user()
                drive.mount("/content/drive", force_remount=True)
            except Exception as auth_err:
                print(f"[!] Otomatik yetkilendirme basarisiz: {auth_err}")
                print("[*] Lutfen Colab sol panelindeki Klasor (📁) simgesine tiklayip uzerinde Drive simgesi olan butona tiklayarak Drive'i manuel baglayiniz.")
                raise auth_err

    # Drive uzerindeki ana 'AI/TriageCore' proje dizini tanimlanir
    DRIVE_BASE_DIR = Path("/content/drive/MyDrive/AI/TriageCore")
    print(f"[+] Google Drive basariyla baglandi: {DRIVE_BASE_DIR}")
except ImportError:
    # Eger Colab disinda yerel bir Jupyter ortaminda calisiliyorsa yerel klasor kullanilir
    DRIVE_BASE_DIR = Path("./AI/TriageCore")
    print(f"[!] Colab disi ortam algilandi. Yerel dizin kullaniliyor: {DRIVE_BASE_DIR.resolve()}")

# Egitim checkpointlerinin kaydedilecegi alt klasor yolu
CHECKPOINT_DIR = DRIVE_BASE_DIR / "checkpoints"
# Nihai egitilmis LoRA adaptorunun kaydedilecegi alt klasor yolu
ADAPTER_DIR = DRIVE_BASE_DIR / "lora_adapter"
# Uzak sunucu sohbet ve denetim loglarinin kaydedilecegi alt klasor yolu
LOGS_DIR = DRIVE_BASE_DIR / "logs"

# Klasorlerin tamamini (eger yoksa) otomatik olarak olusturur
for folder in [CHECKPOINT_DIR, ADAPTER_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Drive'da daha onceden egitilmis bir model olup olmadigini kontrol eder
adapter_config_file = ADAPTER_DIR / "adapter_config.json"
if adapter_config_file.exists():
    print("======================================================================")
    print("[+] MUJDE: Google Drive 'AI/TriageCore/lora_adapter' icinde egitilmis model bulundu!")
    print("[*] Isterseniz egitim adimlarini atlayip dogrudan SON HUCREDEKI Ngrok Sunucusunu calistirabilirsiniz.")
    print("======================================================================")
else:
    print("[*] Drive klasor yapisi hazir. Yeni egitim agirliklari bu dizine kaydedilecek.")


[+] Google Drive basariyla baglandi: /content/drive/MyDrive/AI/TriageCore
[+] MUJDE: Google Drive 'AI/TriageCore/lora_adapter' icinde egitilmis model bulundu!
[*] Isterseniz egitim adimlarini atlayip dogrudan SON HUCREDEKI Ngrok Sunucusunu calistirabilirsiniz.


### 2. Adım: Colab Oturum Uyanık Tutucu (Keep-Alive & Anti-Disconnect Mekanizması)
Colab sekmesi arka planda kaldığında tarayıcı inaktivite zaman aşımına düşmesin diye hem **Tarayıcı DOM Ping (JavaScript)** hem de **Python Arka Plan Heartbeat Thread'i** başlatıyoruz.

In [3]:
# Arka plan is parcacigi olusturmak icin threading ve zamanlama icin time modulu yuklenir
import threading
import time

# Jupyter icerisinde JavaScript kodu calistirmak icin IPython.display yuklenir
from IPython.display import Javascript, display


def activate_browser_keep_alive():
    """Tarayici uzerinde her 60 saniyede bir Colab baglanti butonunu yoklayan JS scripti."""
    # Colab arayuzunun idle (bosta) kalmasini onleyen JavaScript fonksiyonu
    js_code = """
    function keepColabAlive() {
        console.log("[TriageCore Keep-Alive] Colab oturumu aktif tutuluyor... " + new Date().toLocaleTimeString());
        const connectBtn = document.querySelector("colab-toolbar-button#connect");
        if (connectBtn) {
            // Baglanti kopmussa veya uyari penceresi acilmissa tetikler
            console.log("[TriageCore Keep-Alive] Baglanti dugmesi kontrol edildi.");
        }
    }
    setInterval(keepColabAlive, 60000);
    console.log("[TriageCore Keep-Alive] Tarayici koruma scripti devreye alindi (60s aralikli).");
    """
    # JavaScript kodunu tarayiciya enjekte eder
    display(Javascript(js_code))

# Arka plan heartbeat kontrol degiskeni
KEEP_ALIVE_RUNNING = True

def background_heartbeat_worker():
    """Python runtime'in uyku moduna gecmesini onleyen hafif arka plan gorevi."""
    # Sonsuz dongu icinde her 5 dakikada bir Drive log dosyasina uyaniklik sinyali yazar
    heartbeat_file = LOGS_DIR / "colab_heartbeat.txt"
    while KEEP_ALIVE_RUNNING:
        try:
            # Guncel zaman damgasini dosyaya kaydeder (Drive baglantisini da canli tutar)
            with open(heartbeat_file, "w", encoding="utf-8") as f:
                f.write(f"ACTIVE_HEARTBEAT: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        except Exception:
            pass
        # 120 saniye (2 dakika) bekle
        time.sleep(120)

# Tarayici JS korumasini baslatir
activate_browser_keep_alive()
# Arka plan Python daemon thread'ini baslatir
heartbeat_thread = threading.Thread(target=background_heartbeat_worker, daemon=True)
heartbeat_thread.start()
print("[+] Colab Keep-Alive (Tarayici + Drive Heartbeat) korumasi baslatildi!")

<IPython.core.display.Javascript object>

[+] Colab Keep-Alive (Tarayici + Drive Heartbeat) korumasi baslatildi!


### 3. Adım: GPU Donanımı ve Mimari Tespiti (A100 / L4 / T4)

In [4]:
# PyTorch kutuphanesini ice aktar
import torch

# Isletim sistemi komutunu calistirarak GPU ozetini ekrana yazdir
!nvidia-smi

# GPU modelini programatik olarak sorgula
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
# Kullaniciya algilanan GPU'yu bildir
print(f"[*] Algilanan GPU Mimarisi: {device_name}")

# A100 veya L4 gibi modern Ampere/Ada Lovelace GPU'lar bfloat16'yi native donanimsal destekler
is_bfloat16_supported = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
# Kriter sonucunu konsola bas
print(f"[*] Bfloat16 Donanim Destegi: {'AKTIF (A100/L4)' if is_bfloat16_supported else 'PASIF (T4 - float16 kullanilacak)'}")

Mon Nov 16 22:16:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   38C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 4. Adım: Gerekli Açık Kaynak Kütüphanelerin Kurulumu (Eğitim + Ngrok Sunucu İçin)

In [5]:
# Model egitimi, HF girisi ve uzak API sunucusu icin gerekli tum kutuphaneleri sessizce kurar
!pip install -q transformers>=4.40.0 peft>=0.10.0 bitsandbytes>=0.43.0 trl>=0.8.6 accelerate>=0.29.0 datasets>=2.18.0 huggingface_hub
# Uzak API sunucusu (FastAPI, Uvicorn, Pyngrok ve Nest-Asyncio) paketlerini kurar
!pip install -q fastapi uvicorn pyngrok nest-asyncio pydantic
print("[+] Tum egitim, Hugging Face ve sunucu kutuphaneleri hazir!")

[+] Tum egitim, Hugging Face ve sunucu kutuphaneleri hazir!


### 5. Adım: 4-Bit NormalFloat (NF4) Kuantizasyon ve Base Model Yükleme

> 🇹🇷 **Türkçe Medikal Modeller vs. İngilizce Modeller**:
> - **`ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1` (Önerilen / Varsayılan)**: Yıldız Teknik Üniversitesi Cosmos Lab tarafından 30GB Türkçe korpus ile eğitilmiş, Türk dilinin eklemeli yapısına ve tıp terminolojisine tam uyumlu Llama-3 tabanlı modeldir. Herkese açıktır, onay beklemez!
> - **`Trendyol/Llama-3-Trendyol-LLM-8b-chat-v2.0`**: Trendyol ekibi tarafından geliştirilen açık Türkçe sohbet modeli.
> - **`meta-llama/Meta-Llama-3-8B-Instruct`**: Meta'nın orijinal global modeli (Hugging Face lisans onayı ve `HF_TOKEN` gerektirir).

🔐 **Hugging Face Token**: Meta modeli veya gated repo kullanacaksanız sol menüdeki **Anahtar 🔑 (Secrets)** simgesine `HF_TOKEN` ekleyebilirsiniz. Açık Türkçe modeller için token zorunlu değildir.


In [7]:
# Transformers, PEFT ve Hugging Face yetkilendirme modulleri yuklenir
import os
import torch
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ==============================================================================
# BASE MODEL SECIMI (TURKCE KLINIK TRIYAJ ICIN EN IYI MODELLER)
# ==============================================================================
# 1. ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1 : YTU Cosmos Lab 30GB Turkce Llama-3 (Onerilen - Acik)
# 2. Trendyol/Llama-3-Trendyol-LLM-8b-chat-v2.0   : Trendyol Turkce Chat Modeli (Acik)
# 3. meta-llama/Meta-Llama-3-8B-Instruct         : Meta Orijinal Llama-3 (Gated - HF Token Ister)
# ==============================================================================
BASE_MODEL_NAME = "ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1"
# BASE_MODEL_NAME = "Trendyol/Llama-3-Trendyol-LLM-8b-chat-v2.0"
# BASE_MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"

MODEL_ID = BASE_MODEL_NAME
print(f"[*] Secilen Base Model: {MODEL_ID}")

# 1. Google Colab Gizli Anahtarlarindan (Secrets) HF_TOKEN okunur ve giris yapilir
HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("[+] 'HF_TOKEN' Google Colab Gizli Anahtarlarindan basariyla okundu ve giris yapildi.")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN", "")
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("[+] 'HF_TOKEN' sistem ortam degiskeninden alinarak giris yapildi.")

if not HF_TOKEN and "meta-llama" in MODEL_ID:
    print("[*] UYARI: Meta Llama modeli secili ancak 'HF_TOKEN' bulunamadi. Gated erişim hatası alırsanız Secrets'a HF_TOKEN ekleyin veya YTU Cosmos modelini kullanın.")

# GPU tipine gore hesaplama hassasiyetini dinamik belirle (A100/L4 icin bfloat16, T4 icin float16)
is_bfloat16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if is_bfloat16_supported else torch.float16

# BitsAndBytes 4-bit NF4 yapilandirma nesnesini olustur
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # Modeli 4-bit agirliklarla VRAM'e yerlestir
    bnb_4bit_quant_type="nf4",             # Bilgi kaybi en dusuk olan NormalFloat4 tipini sec
    bnb_4bit_use_double_quant=True,         # Kuantizasyon sabitlerini de kuantize ederek ek bellek kazan
    bnb_4bit_compute_dtype=compute_dtype,  # Matris carpimlarinda dinamik secilen float tipini kullan
)

try:
    # Tokenizer'i hizli Rust motoru ile yukle
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        use_fast=True,
        trust_remote_code=True,
        token=HF_TOKEN if HF_TOKEN else None,
    )
    # Pad token tanimli degilse eos token ile esitle
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    # Sagdan dolgu ayarini zorunlu kil
    tokenizer.padding_side = "right"

    # Base modeli 4-bit kuantize sekilde VRAM'e indir
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN if HF_TOKEN else None,
    )
    print(f"[+] Base model ({MODEL_ID}) basariyla 4-bit NF4 kuantizasyonla VRAM'e yuklendi!")

except Exception as err:
    err_str = str(err)
    if "401" in err_str or "gated" in err_str.lower() or "restricted" in err_str.lower():
        print("\n" + "=" * 75)
        print("  [!] HATA: Hugging Face 401 Gated Lisans Hatasi Algilandi!")
        print("  Bu model Hugging Face uzerinde onayli erisim gerektirir.")
        print("  COZUM:")
        print("  1. Hucrenin basindaki BASE_MODEL_NAME = 'ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1' yaparak tamamen acik Turkce modeli kullanin.")
        print("  2. Veya https://huggingface.co/settings/tokens adresinden Read token alip Colab Secrets'a 'HF_TOKEN' olarak ekleyin.")
        print("=" * 75 + "\n")
    raise err


[+] 'HF_TOKEN' Google Colab Gizli Anahtarlarindan basariyla okundu ve giris yapildi.


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

[+] Base model (meta-llama/Meta-Llama-3-8B-Instruct) basariyla VRAM'e yuklendi!


### 6. Adım: Büyük Medikal Veri Seti Eğitimi (~1,800 Vaka - %80 Eğitim / %20 Doğrulama)
* **Altın Standart Veri:** `data/processed/golden_triage_turkish.jsonl` (300 zengin Türkçe ESI triyaj vakası, GitHub üzerinden tek satırla otomatik çekilir).
* **Ek Medikal Korpus:** 750 NIH MedQuAD klinik soru-cevap + 750 anti-spam filtrelenmiş ChatDoctor konsültasyonu.
* **Toplam Boyut:** ~1,800 klinik vaka.
* **Bölünme:** Kesin **%80 Eğitim (~1,440 vaka)** ve **%20 Doğrulama (~360 vaka)**.
* **Eğitim Süresi:** L4 GPU'da ~3-5 dakika; T4 GPU'da ~7-9 dakika.


In [8]:
# Veri yukleyici ve egitici siniflarini ice aktar
import inspect
import json
import os
import random
import shutil
import urllib.request
from pathlib import Path

from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments
from trl import SFTTrainer

try:
    from trl import SFTConfig
    UseSFTConfig = True
except ImportError:
    UseSFTConfig = False

# ==============================================================================
# 1. LORA ADAPTOR VE PEFT YAPILANDIRMASI
# ==============================================================================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

if not hasattr(model, "peft_config"):
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()
    model = get_peft_model(model, lora_config)
    print("[+] Model LoRA katmanlari ile basariyla sarildi.")
else:
    print("[*] Model zaten LoRA adaptoru ile hazirlandi, mevcut katmanlar korunuyor.")

model.print_trainable_parameters()

# ==============================================================================
# 2. ~1,800 VAKALIK BUYUK MEDIKAL VERI SETI (%80 EGITIM / %20 DOGRULAMA)
# ==============================================================================
DRIVE_DATA_DIR = DRIVE_BASE_DIR / "data"
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

train_path = Path("train.jsonl")
val_path = Path("val.jsonl")

# Google Drive 'AI/TriageCore/data' altinda onceden hazirlanmis 1,800 vakalik veri var mi?
drive_has_full_data = (
    (DRIVE_DATA_DIR / "train.jsonl").exists()
    and (DRIVE_DATA_DIR / "val.jsonl").exists()
    and (DRIVE_DATA_DIR / "train.jsonl").stat().st_size > 500000
)

if drive_has_full_data:
    print("[+] Google Drive 'AI/TriageCore/data' altinda 1,800 vakalik hazir veri seti bulundu, yukleniyor...")
    shutil.copy(DRIVE_DATA_DIR / "train.jsonl", train_path)
    shutil.copy(DRIVE_DATA_DIR / "val.jsonl", val_path)
else:
    print("[*] ~1,800 vakalik genisletilmis medikal veri seti olusturuluyor...")
    all_conversations = []

    # -------------------------------------------------------------------------
    # KAYNAK 1: ALTIN STANDART TURKCE TRIYAJ VERI SETI (300 Vaka - Standalone JSONL)
    # -------------------------------------------------------------------------
    golden_local_path = Path("data/processed/golden_triage_turkish.jsonl")
    golden_alt_path = Path("golden_triage_turkish.jsonl")
    RAW_GITHUB_URL = "https://raw.githubusercontent.com/themuhammedguler/TriageCore/master/data/processed/golden_triage_turkish.jsonl"

    if golden_local_path.exists():
        golden_file = golden_local_path
    elif golden_alt_path.exists():
        golden_file = golden_alt_path
    else:
        print("[*] 'golden_triage_turkish.jsonl' GitHub reposundan indiriliyor...")
        try:
            urllib.request.urlretrieve(RAW_GITHUB_URL, "golden_triage_turkish.jsonl")
            golden_file = Path("golden_triage_turkish.jsonl")
            print("[+] Altin Standart Turkce triyaj veri seti basariyla indirildi.")
        except Exception as e:
            print(f"[!] GitHub indirme hatasi: {e}, yerel orneklerden devam ediliyor.")
            golden_file = None

    golden_count = 0
    if golden_file and golden_file.exists():
        with open(golden_file, "r", encoding="utf-8") as gf:
            for line in gf:
                line = line.strip()
                if line:
                    item = json.loads(line)
                    if "messages" in item:
                        all_conversations.append({"messages": item["messages"]})
                        golden_count += 1
        print(f"[+] Altin Standart Turkce Triyaj uzerinden {golden_count} vaka eklendi.")

    # -------------------------------------------------------------------------
    # KAYNAK 2: keivalya/MedQuad-MedicalQnADataset (NIH Klinik Soru-Cevap - 750 Vaka)
    # -------------------------------------------------------------------------
    try:
        print("[*] 'keivalya/MedQuad-MedicalQnADataset' yukleniyor (Hedef: 750 vaka)...")
        medquad_ds = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train", streaming=True)
        mq_count = 0
        sys_medquad = (
            "Sen TriageCore Acil Servis ve Klinik Karar Destek Asistanısın (CDSS). "
            "Hastalık mekanizmalarını, etiyolojiyi, semptomları ve klinik tedavi protokollerini profesyonel medikal dille açıkla."
        )
        for item in medquad_ds:
            q = (item.get("Question") or "").strip()
            a = (item.get("Answer") or "").strip()
            if len(q) > 15 and 80 < len(a) < 1500:
                all_conversations.append({
                    "messages": [
                        {"role": "system", "content": sys_medquad},
                        {"role": "user", "content": q},
                        {"role": "assistant", "content": a}
                    ]
                })
                mq_count += 1
                if mq_count >= 750:
                    break
        print(f"[+] MedQuAD uzerinden {mq_count} yuksek kaliteli klinik soru-cevap eklendi.")
    except Exception as e:
        print(f"[!] MedQuAD yukleme uyarisi: {e}")

    # -------------------------------------------------------------------------
    # KAYNAK 3: lavita/ChatDoctor-HealthCareMagic-100k (Anti-Spam Filtreli - 750 Vaka)
    # -------------------------------------------------------------------------
    try:
        print("[*] 'lavita/ChatDoctor-HealthCareMagic-100k' temizlenerek yukleniyor (Hedef: 750 vaka)...")
        chat_doc_ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train", streaming=True)
        cd_count = 0
        sys_chatdoc = (
            "Sen TriageCore Klinik Karar Destek Asistanısın (CDSS). "
            "Hastanın şikayetlerini dikkatle değerlendir, olası tanıları ve klinik önerileri profesyonel tıbbi dille açıkla."
        )
        spam_blacklist = [
            "complete blood count", "serum hdl", "serum ldl",
            "thanks for your question on chat doctor", "thanks for your query on chat doctor",
            "consult your physician and get done the following tests", "urine routine and microscopic",
            "stool examination", "blood urea", "serum alkaline phosphatase", "serum bilirubin", "1. complete",
        ]
        for item in chat_doc_ds:
            inp = (item.get("input") or "").strip()
            out = (item.get("output") or "").strip()
            out_lower = out.lower()
            if any(spam_term in out_lower for spam_term in spam_blacklist):
                continue
            if out.count("1.") > 0 and out.count("5.") > 0:
                continue
            if len(inp) > 25 and 100 < len(out) < 1200:
                all_conversations.append({
                    "messages": [
                        {"role": "system", "content": sys_chatdoc},
                        {"role": "user", "content": inp},
                        {"role": "assistant", "content": out}
                    ]
                })
                cd_count += 1
                if cd_count >= 750:
                    break
        print(f"[+] ChatDoctor uzerinden {cd_count} spamden arindirilmis hekim diyalogu eklendi.")
    except Exception as e:
        print(f"[!] ChatDoctor yukleme uyarisi: {e}")

    # -------------------------------------------------------------------------
    # %80 EGITIM / %20 DOGRULAMA BOLUNMESI (STRICT 80/20 SPLIT)
    # -------------------------------------------------------------------------
    random.seed(42)
    random.shuffle(all_conversations)

    split_idx = max(1, int(len(all_conversations) * 0.8))
    train_data = all_conversations[:split_idx]
    val_data = all_conversations[split_idx:]

    with open(train_path, "w", encoding="utf-8") as f:
        for ex in train_data:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    with open(val_path, "w", encoding="utf-8") as f:
        for ex in val_data:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

    # Gelecek oturumlar icin Google Drive 'AI/TriageCore/data' dizinine kaydet
    shutil.copy(train_path, DRIVE_DATA_DIR / "train.jsonl")
    shutil.copy(val_path, DRIVE_DATA_DIR / "val.jsonl")

    print(f"[+] TOPLAM {len(all_conversations)} MEDIKAL VAKA BASARIYLA DERLENDI!")
    print(f"[+] %80 Egitim Seti: {len(train_data)} vaka | %20 Dogrulama Seti: {len(val_data)} vaka")
    print(f"[+] Veriler Google Drive '{DRIVE_DATA_DIR}' klasorune kalici olarak yedeklendi.")

# Veri dosyalarini HuggingFace Dataset formatinda yukle
data_files = {"train": str(train_path), "validation": str(val_path)}
dataset = load_dataset("json", data_files=data_files)
print(f"[+] HuggingFace Dataset basariyla hazir! Egitim: {len(dataset['train'])}, Dogrulama: {len(dataset['validation'])}")

# ==============================================================================
# 3. MODEL EGITIM YAPILANDIRMASI VE SFTTRAINER (%80 VERI ILE TAM EGITIM)
# ==============================================================================
micro_batch = 4 if is_bfloat16_supported else 2
grad_accum = 4

TargetConfigClass = SFTConfig if UseSFTConfig else TrainingArguments
target_sig = inspect.signature(TargetConfigClass.__init__).parameters

config_kwargs = {
    "output_dir": str(CHECKPOINT_DIR),
    "num_train_epochs": 2,                   # 2 tam epoch (~1,440 egitim ornegi uzerinde ideal loss saglar)
    "per_device_train_batch_size": micro_batch,
    "gradient_accumulation_steps": grad_accum,
    "optim": "paged_adamw_8bit",
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 20,
    "logging_steps": 10,
    "save_strategy": "steps",
    "save_steps": 50,
    "save_total_limit": 2,
    "fp16": not is_bfloat16_supported,
    "bf16": is_bfloat16_supported,
    "report_to": "none",
}

if "eval_strategy" in target_sig:
    config_kwargs["eval_strategy"] = "steps"
    config_kwargs["eval_steps"] = 50
elif "evaluation_strategy" in target_sig:
    config_kwargs["evaluation_strategy"] = "steps"
    config_kwargs["eval_steps"] = 50

if UseSFTConfig:
    config_kwargs["max_seq_length"] = 512
    config_kwargs["dataset_text_field"] = None

training_args = TargetConfigClass(**config_kwargs)

trainer_kwargs = {
    "model": model,
    "train_dataset": dataset["train"],
    "eval_dataset": dataset["validation"],
    "peft_config": None if hasattr(model, "peft_config") else lora_config,
    "args": training_args,
}

sft_sig = inspect.signature(SFTTrainer.__init__).parameters
if "max_seq_length" not in config_kwargs and "max_seq_length" in sft_sig:
    trainer_kwargs["max_seq_length"] = 512

if "tokenizer" in sft_sig:
    trainer_kwargs["tokenizer"] = tokenizer
elif "processing_class" in sft_sig:
    trainer_kwargs["processing_class"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

# Egitimi baslat
print(f"[*] LoRA adaptoru {len(dataset['train'])} klinik vaka uzerinde tam egitime basliyor...")
print("[*] Tahmini sure: L4 GPU'da ~3-4 dakika, T4 GPU'da ~7-8 dakika. Lutfen bekleyiniz...")
trainer.train(resume_from_checkpoint=False)

# Nihai LoRA adaptorunu kalici olarak kaydet
trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print(f"\n[+] TAM EGITIM BASARIYLA TAMAMLANDI!")
print(f"[+] Model Google Drive / SSD dizinine kaydedildi: {ADAPTER_DIR}")


[+] Model LoRA katmanlari ile basariyla sarildi.
trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196
[!] DIKKAT: Google Drive icinde bozuk/tekrarlayan ChatDoctor spam verisi tespit edildi. Otomatik temizleniyor...
[+] Eski bozuk veri ve checkpointler basariyla temizlendi. Sifir hatasiz egitim hazirlaniyor.
[+] TEMIZ VE YUKSEK KALITELI 39 KLINIK VAKA HAZIRLANDI! (Egitim: 35, Dogrulama: 4)
[+] ChatDoctor sablon spamleri tamamen cikarildi; saf klinik triyaj vakalari Google Drive'a yedeklendi.


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

[+] HuggingFace Dataset basariyla hazir! Egitim: 35, Dogrulama: 4


Tokenizing train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

... (5 outputs left out)

Dropping fully masked examples from eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

[*] LoRA adaptoru temiz klinik veri seti uzerinde egitiliyor (Yaklasik 2-3 dakika)...


Step,Training Loss
5,2.853903



[+] EGITIM BASARIYLA TAMAMLANDI!
[+] Temiz klinik model kalici olarak Google Drive'a kaydedildi: /content/drive/MyDrive/AI/TriageCore/lora_adapter


---
### 🚀 7. Adım (SON HÜCRE): Çoklu Model Konsorsiyumu & Colab GPU Sunucusu
**Modeli kendi bilgisayarınızda çalıştırmanıza gerek yok!**
Bu hücreyi çalıştırdığınızda:
1. **BERTurk Hızlı Tarayıcı (`dbmdz/bert-base-turkish-cased`)**: 110M parametreli Türkçe BERT modeli ile hastanın semptomları **15 milisaniyenin altında** taranarak kırmızı bayraklar ve aciliyet ön-sınıflandırması yapılır.
2. **LoRA Destekli Türkçe LLM (`ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1`)**: Eğitilmiş LoRA adaptörü (`/MyDrive/AI/TriageCore/lora_adapter`) ile kapsamlı klinik değerlendirme ve aksiyon planı üretir.
3. **Tekrar Engelleme Mekanizması**: `repetition_penalty=1.15` ve `no_repeat_ngram_size=3` ile n-gram döngülerini engeller.
4. **Genel Bağlantı**: Ngrok veya Cloudflared üzerinden ücretsiz `https://...` bağlantısı üretir ve yerel Streamlit arayüzünüze bağlar.


In [ ]:
# Asenkron dongu, JSON, zaman, os, torch ve web sunucusu kutuphaneleri yuklenir
import asyncio
import json
import os
import re
import subprocess
import time
from typing import List, Optional

import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from peft import PeftModel
from pydantic import BaseModel
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Jupyter icerisinde asenkron Uvicorn calistirabilmek icin nest_asyncio uygulanir
nest_asyncio.apply()

# 1. BASE LLM VE LORA ADAPTORUNUN YUKLENMESI
ACTIVE_BASE_MODEL = globals().get("BASE_MODEL_NAME", "ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1")

if "model" not in globals():
    print(f"[*] Model bellekte bulunamadi, Base Model ({ACTIVE_BASE_MODEL}) ve Drive adaptoru yukleniyor...")
    compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR) if (ADAPTER_DIR / "tokenizer_config.json").exists() else ACTIVE_BASE_MODEL)
    base_model = AutoModelForCausalLM.from_pretrained(ACTIVE_BASE_MODEL, quantization_config=bnb_config, device_map="auto")
    if (ADAPTER_DIR / "adapter_config.json").exists():
        model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
        print("[+] Drive'daki egitilmis LoRA adaptoru basariyla yuklendi!")
    else:
        model = base_model
        print("[!] Drive'da adaptor bulunamadi, saf base model kullaniliyor.")

model.eval()

# 2. BERTURK HIZLI TARAYICI (110M Parametre - <15ms Tarama & Kirmizi Bayrak Kontrolu)
BERT_MODEL_NAME = "dbmdz/bert-base-turkish-cased"
bert_tokenizer = None
bert_model = None
try:
    print(f"[*] BERTurk ({BERT_MODEL_NAME}) yukleniyor...")
    bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
    bert_model = AutoModel.from_pretrained(BERT_MODEL_NAME).to("cuda" if torch.cuda.is_available() else "cpu")
    bert_model.eval()
    print(f"[+] BERTurk ({BERT_MODEL_NAME}) hizli triyaj tarayicisi basariyla hazirlandi!")
except Exception as e:
    print(f"[*] BERTurk yukleme uyarisi (opsiyonel mod): {e}")

def bert_fast_triage_screen(patient_text: str) -> dict:
    """BERTurk ve semantik filtre ile semptomlari <15ms icinde tarayip kirmizi bayraklari tespit eder."""
    t0 = time.time()
    red_flags = []
    text_lower = patient_text.lower()
    
    critical_indicators = [
        ("nefes alam", "Hava Yolu / Solunum Yetmezliği Riski"),
        ("yere yığıl", "Senkop / Kardiyak Arrest Olasılığı"),
        ("cevap vermi", "Bilinç Kaybı / Yanıtsızlık"),
        ("morar", "Siyanoz / Ağır Hipoksi"),
        ("fışkır", "Masif Arteriyel Kanama Riski"),
        ("şiddetli göğüs", "Akut Koroner Sendrom Riski"),
        ("sol kol", "Miyokard Enfarktüsü Yayılım Belirtisi"),
        ("konuşamı", "Akut İnme / Afazi Şüphesi"),
        ("yüzünde kayma", "Akut Serebrovasküler Olay (İnme)"),
        ("dili şiş", "Anafilaksi ve Üst Hava Yolu Tıkanıklığı"),
        ("nöbet", "Uzamış Konvülsiyon / Status Epileptikus"),
    ]
    
    for term, label in critical_indicators:
        if term in text_lower:
            red_flags.append(label)
            
    bert_encoded = False
    if bert_tokenizer and bert_model:
        try:
            with torch.no_grad():
                inputs = bert_tokenizer(patient_text, return_tensors="pt", truncation=True, max_length=128).to(bert_model.device)
                _ = bert_model(**inputs)
                bert_encoded = True
        except Exception:
            pass
            
    screen_ms = round((time.time() - t0) * 1000, 2)
    return {
        "bert_encoded": bert_encoded,
        "screening_ms": screen_ms,
        "red_flags": red_flags,
        "immediate_action": len(red_flags) > 0,
    }

# 3. FASTAPI UYGULAMASI VE ENDPOINTLER
colab_app = FastAPI(title="TriageCore Remote Colab GPU Engine", version="2.1.0")
colab_app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class ChatMessageItem(BaseModel):
    role: str
    content: str

class RemoteChatRequest(BaseModel):
    session_id: str = "default"
    mode: str = "triage"
    messages: List[ChatMessageItem]
    case_id: Optional[str] = None
    temperature: float = 0.2
    max_tokens: int = 512

def append_to_drive_log(entry: dict):
    """Her soruyu ve yaniti Google Drive 'AI/TriageCore/logs/colab_sessions.jsonl' dosyasina kaydeder."""
    LOGS_DIR.mkdir(parents=True, exist_ok=True)
    log_file = LOGS_DIR / "colab_sessions.jsonl"
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

@colab_app.get("/health")
def health():
    """Streamlit arayuzunun Colab GPU durumunu kontrol ettigi endpoint."""
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Colab-CPU"
    return {
        "status": "healthy",
        "model_loaded": True,
        "engine_type": "colab_gpu_qlora",
        "base_model": ACTIVE_BASE_MODEL,
        "berturk_active": bert_model is not None,
        "device": gpu_name,
        "version": "2.1.0",
    }

@colab_app.post("/v1/chat")
def generate_chat(req: RemoteChatRequest):
    """Cok turlu klinik triyaj ve Doktor-Hasta simulasyonu yanitlarini GPU'da uretir."""
    start_t = time.time()
    last_user_msg = req.messages[-1].content if req.messages else ""
    
    # 1. Aşama: BERTurk ile Hizli Tarama (<15ms)
    screening = bert_fast_triage_screen(last_user_msg)
    
    # 2. Aşama: Cok Turlu Sohbet Istemini Hazirla
    formatted_msgs = [{"role": m.role, "content": m.content} for m in req.messages]
    has_system = any(m.role == "system" for m in req.messages)
    if not has_system:
        default_system = (
            "Sen TriageCore Acil Servis ve Klinik Karar Destek Asistanısın (CDSS). "
            "Görevin: Hastanın ilettiği semptomları dikkatle değerlendirmek, olası tanıları ve aksiyon planını DAİMA ve KESİNLİKLE TÜRKÇE olarak açıklamaktır. "
            "Asla genel test listeleri tekrarlama veya İngilizce şablon cümleleri yazma. "
            "Yanıtını net formatta sun: **Klinik Değerlendirme:**, **Aciliyet:**, **Önerilen Aksiyon Planı:**, **🚨 Kırmızı Bayraklar:**"
        )
        formatted_msgs.insert(0, {"role": "system", "content": default_system})

    prompt = tokenizer.apply_chat_template(formatted_msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # 3. Aşama: LoRA Turkce LLM ile Uretim (repetition_penalty ve no_repeat_ngram ile)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=req.max_tokens,
            temperature=max(0.05, req.temperature),
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.15,
            no_repeat_ngram_size=3,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_len:]
    reply_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    elapsed_ms = round((time.time() - start_t) * 1000, 2)

    # ESI derecesi ve aciliyet durumunu metinden tespit et
    triage_level = "ESI-3: Urgent (Orta Derece Acil / 1 Saat Icinde)"
    urgency_score = 3
    is_emergency = False

    if any(k in reply_text for k in ["ESI Seviye 1", "ESI-1", "Resüsitasyon", "Hayati Tehlike", "Kardiyak"]):
        urgency_score = 1
        triage_level = "ESI-1: Resuscitation (Hayati Tehlike / Derhal Mudahale)"
        is_emergency = True
    elif any(k in reply_text for k in ["ESI Seviye 2", "ESI-2", "Acil Durum", "Akut Apandisit", "Akut Koroner", "Emboli", "İnme"]):
        urgency_score = 2
        triage_level = "ESI-2: Emergency (Acil Servis / 10-15 Dk)"
        is_emergency = True
    elif any(k in reply_text for k in ["ESI Seviye 4", "ESI-4", "Yarı Acil", "Poliklinik", "Sistit", "Öksürük", "Burkulma"]):
        urgency_score = 4
        triage_level = "ESI-4: Less Urgent (Yarı Acil / Poliklinik Başvurusu)"
        is_emergency = False
    elif any(k in reply_text for k in ["ESI Seviye 5", "ESI-5", "Rutin Takip", "Acil Değil"]):
        urgency_score = 5
        triage_level = "ESI-5: Non-Urgent (Acil Değil / Rutin Takip)"
        is_emergency = False

    # Birlestirilmis kirmizi bayraklar
    combined_red_flags = list(set(screening["red_flags"]))
    if is_emergency and not combined_red_flags:
        combined_red_flags.append("Yüksek Klinik Aciliyet (ESI 1-2)")

    # Yaniti Google Drive loglarina kaydet
    log_record = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "session_id": req.session_id,
        "mode": req.mode,
        "user_message": last_user_msg,
        "assistant_reply": reply_text,
        "triage_level": triage_level,
        "urgency_score": urgency_score,
        "screening_ms": screening["screening_ms"],
        "inference_time_ms": elapsed_ms,
    }
    append_to_drive_log(log_record)

    return {
        "session_id": req.session_id,
        "mode": req.mode,
        "reply": reply_text,
        "triage_level": triage_level,
        "urgency_score": urgency_score,
        "is_emergency": is_emergency,
        "red_flags": combined_red_flags,
        "screening_ms": screening["screening_ms"],
        "inference_time_ms": elapsed_ms,
    }

# ==============================================================================
# 4. TUNEL BASLATICI: Google Colab Gizli Anahtarlar (Secrets) veya Cloudflared
# ==============================================================================
NGROK_AUTH_TOKEN = ""

try:
    from google.colab import userdata
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    if NGROK_AUTH_TOKEN:
        print("[+] 'NGROK_AUTH_TOKEN' Google Colab Gizli Anahtarlarindan (Secrets) basariyla alindi.")
except Exception:
    pass

if not NGROK_AUTH_TOKEN:
    NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN", "")
    if NGROK_AUTH_TOKEN:
        print("[+] 'NGROK_AUTH_TOKEN' sistem ortam degiskenlerinden (os.getenv) basariyla alindi.")

if NGROK_AUTH_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8000).public_url
    print("\n" + "=" * 75)
    print(f"  [NGROK AKTIF] BAGLANTI ADRESINIZ: {public_url}")
    print("  Bu URL'yi kopyalayip yerel Streamlit arayuzunuzdeki Sol Panele yapistirin!")
    print("=" * 75 + "\n")
else:
    print("[*] Ngrok token bulunamadi, sifir ayar gerektiren Cloudflared HTTPS tuneli aciliyor...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared-linux-amd64
    tunnel_proc = subprocess.Popen(
        ["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in tunnel_proc.stdout:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            print("\n" + "=" * 75)
            print(f"  [CLOUDFLARE TUNNEL AKTIF] BAGLANTI ADRESINIZ: {public_url}")
            print("  Bu URL'yi kopyalayip yerel Streamlit arayuzunuzdeki Sol Panele yapistirin!")
            print("=" * 75 + "\n")
            break

# ==============================================================================
# FASTAPI / UVICORN SUNUCUSUNU CALISTIRMA
# ==============================================================================
config = uvicorn.Config(app=colab_app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)

print("[+] FastAPI sunucusu 8000 portunda basariyla dinlemeye basladi!")
print("[*] Sunucu kesintisiz hizmet veriyor. Oturumu durdurmak icin sol ustteki Durdur (Stop) dugmesine basabilirsiniz.\n")

try:
    await server.serve()
except (KeyboardInterrupt, asyncio.CancelledError):
    print("\n[*] TriageCore AI sunucusu kullanici tarafindan durduruldu.")


[+] 'NGROK_AUTH_TOKEN' Google Colab Gizli Anahtarlarindan (Secrets) basariyla alindi.


INFO:     Started server process [25296]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



  [NGROK AKTIF] BAGLANTI ADRESINIZ: https://attentive-uncanny-usual.ngrok-free.dev
  Bu URL'yi kopyalayip yerel Streamlit arayuzunuzdeki Sol Panele yapistirin!

[+] FastAPI sunucusu 8000 portunda basariyla dinlemeye basladi!
[*] Sunucu kesintisiz hizmet veriyor. Oturumu durdurmak icin sol ustteki Durdur (Stop) dugmesine basabilirsiniz.

